## Task 18: Summary of Synthetic Data Generation and Evaluation

**1. Steps Taken:**

*   Loaded and preprocessed `fraud_transactions.csv`.
*   Trained a TVAESynthesizer model.
*   Generated 5000 synthetic records.
*   Evaluated fidelity (KS test, correlation), utility (TSTR with RandomForestClassifier), and privacy (nearest-neighbor distances).

**2. Results and Evaluation:**

*   **Fidelity:** Moderate difference in numerical feature ('amt') distribution (KS stat: 0.165). Correlation comparison was trivial for a single numerical feature.
*   **Utility:** Synthetic data provided good utility (AUC 0.9347) for fraud prediction, though with a notable drop in F1-score (0.6081) compared to the real-trained baseline (AUC 0.9831, F1 0.8343).
*   **Privacy:** Significant memorization detected (minimum and 25th percentile NN distance of 0.0), indicating a high privacy risk.

**Overall Conclusion:** TVAE-generated synthetic fraud data showed reasonable utility but had limitations in fidelity and critical privacy concerns due to memorization, necessitating further improvements for sensitive applications.

In this notebook, we will evaluate two techniques to generate structured synthetic data: **Tabular GAN** and **Tabular Variational Autoencoder**.

#Part 1: Tabular GANs
Tabular GANs are a type of Generative Adversarial Network (GAN) specifically designed to generate synthetic tabular data (data organized in rows and columns, like a spreadsheet or a Pandas DataFrame) that closely resembles a real-world dataset. Traditional GANs were initially more successful in generating continuous data like images. Tabular data presents unique challenges due to the presence of:
* Mixed Data Types: Tables often contain both numerical (continuous or discrete) and categorical features.
* Complex Correlations: Features in a table can have intricate linear and non-linear relationships.
* Unbalanced Categories: Categorical features can have classes with highly varying frequencies.
* Discrete Values: Even numerical columns might represent discrete quantities.


CTGAN (Conditional Tabular Generative Adversarial Network) addresses these challenges through several key innovations built upon the standard GAN architecture:
* Generator (G):
Takes random noise as input.
Its goal is to generate synthetic data samples that the discriminator cannot distinguish from real data.
It uses neural networks (typically Multi-Layer Perceptrons or MLPs) to transform the noise into synthetic tabular data.
* Discriminator (D):
Takes a batch of data as input, which can be a mix of real data samples from the original dataset and synthetic data samples generated by the generator.
Its goal is to correctly classify each input sample as either "real" or "synthetic."
It also uses neural networks (MLPs) for this classification task.
* Adversarial Training:
The generator and discriminator are trained in an adversarial manner.
The generator tries to fool the discriminator by producing increasingly realistic synthetic data.
The discriminator tries to become better at distinguishing real from synthetic data.
This competition drives both networks to improve, ideally leading the generator to produce synthetic data that is statistically very similar to the real data.


In essence, CTGAN aims to learn the underlying data generation process of  tabular dataset by training a generator to produce synthetic data that fools a discriminator trained to distinguish it from the real data.

In [ ]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors


## 1. Load and inspect `AnguloM/loan_data`

The dataset is a LendingClub‑style consumer loan dataset hosted on Hugging Face.  
Key fields (from the dataset card):

- `not.fully.paid`: **outcome** – 1 if the loan was *not* fully repaid (default/charge‑off), 0 otherwise. [web:111]
- `credit.policy`: 1 if the customer meets LendingClub's underwriting criteria.
- `purpose`: loan purpose (debt_consolidation, credit_card, etc.).
- Numeric features: `int.rate`, `installment`, `log.annual.inc`, `dti`, `fico`, `days.with.cr.line`, `revol.bal`, `revol.util`, `inq.last.6mths`, `delinq.2yrs`, `pub.rec`. [web:109][web:142]

We will:
1. Load the dataset.
2. Inspect schema and basic statistics.
3. Confirm class balance of `not.fully.paid`.


In [ ]:
loan_ds = load_dataset("AnguloM/loan_data")
df = loan_ds["train"].to_pandas()

df.head()

README.md: 0.00B [00:00, ?B/s]

loan.zip:   0%|          | 0.00/218k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9578 [00:00<?, ? examples/s]

,credit.policy,purpose,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9578 entries, 0 to 9577
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   credit.policy      9578 non-null   int64  
 1   purpose            9578 non-null   object 
 2   int.rate           9578 non-null   float64
 3   installment        9578 non-null   float64
 4   log.annual.inc     9578 non-null   float64
 5   dti                9578 non-null   float64
 6   fico               9578 non-null   int64  
 7   days.with.cr.line  9578 non-null   float64
 8   revol.bal          9578 non-null   int64  
 9   revol.util         9578 non-null   float64
 10  inq.last.6mths     9578 non-null   int64  
 11  delinq.2yrs        9578 non-null   int64  
 12  pub.rec            9578 non-null   int64  
 13  not.fully.paid     9578 non-null   int64  
dtypes: float64(6), int64(7), object(1)
memory usage: 1.0+ MB


In [ ]:
df["not.fully.paid"].value_counts(normalize=True)


,proportion
not.fully.paid,
0,0.839946
1,0.160054


## 2. Preprocessing with `not.fully.paid` as outcome

Our prediction / label variable is:

- `not.fully.paid` (1 = default / not fully repaid, 0 = fully paid).

We define three groups of columns:

- **Target:** `not.fully.paid`
- **Categorical features:** `purpose`, `credit.policy` (treated as discrete category).
- **Numeric features:** rate, installment, income, FICO, etc.

Steps:
1. Drop rows with missing values (dataset is usually clean, but we are defensive).
2. Split into features `X` and target `y`.


In [ ]:
# Drop any NA rows to simplify the lab
df = df.dropna().reset_index(drop=True)

target_col = "not.fully.paid"

cat_cols = ["purpose", "credit.policy"]
num_cols = [
    "int.rate",
    "installment",
    "log.annual.inc",
    "dti",
    "fico",
    "days.with.cr.line",
    "revol.bal",
    "revol.util",
    "inq.last.6mths",
    "delinq.2yrs",
    "pub.rec"
]

X = df[cat_cols + num_cols].copy()
y = df[target_col].astype(int)

df[[target_col]].value_counts(normalize=True)


,proportion
not.fully.paid,
0,0.839946
1,0.160054


## 3. Train/test split on real data

We will later:

- Train a classifier on **real data** (baseline).
- Train a classifier on **synthetic data** and test on **real data** (TSTR).

To do this, we split the real dataset into `train` and `test` with stratification on `not.fully.paid`.


In [ ]:
real_train, real_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[target_col]
)

real_train.shape, real_test.shape


((7662, 14), (1916, 14))

## 4. Train CTGAN to generate synthetic loans

We will use **CTGAN**, a GAN variant designed specifically for mixed‑type tabular data.

Key design choices:

- Input to CTGAN: all feature columns **plus** the outcome `not.fully.paid`.
- `discrete_columns` parameter: includes all categorical fields and integer count features, including the binary outcome.

This lets us later **condition** on `not.fully.paid` if we want to oversample defaulted loans.


In [ ]:
from ctgan import CTGAN

ctgan_data = df[cat_cols + num_cols + [target_col]].copy()

discrete_cols = cat_cols + ["inq.last.6mths", "delinq.2yrs", "pub.rec", target_col]

ctgan = CTGAN(
    epochs=300,       # increase for better quality if you have time/compute
    batch_size=500,
    verbose=True
)

ctgan.fit(ctgan_data, discrete_columns=discrete_cols)

Gen. (+00.00) | Discrim. (+00.00):   0%|          | 0/300 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Gen. (-01.18) | Discrim. (-00.11): 100%|██████████| 300/300 [02:56<00:00,  1.70it/s]


## 5. Generate synthetic data

Let's generate 10,000 synthetic loan records.

We will inspect:

- Schema (column names, dtypes).
- Class balance for `not.fully.paid` in synthetic data vs. real data.


In [ ]:
n_synth = 10_000
synthetic = ctgan.sample(n_synth)
synthetic.head()


,purpose,credit.policy,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,all_other,0,0.122783,170.474717,10.971221,3.763832,661,2312.606709,12322,52.932843,0,0,0,1
1,credit_card,1,0.184747,308.311688,10.594259,13.463580,672,1899.235965,5139,55.211501,10,0,0,0
2,major_purchase,0,0.212960,167.337289,11.012977,14.577807,665,1546.915687,3155,4.439505,0,0,0,0
3,home_improvement,1,0.078611,265.265010,10.931548,1.722126,793,2564.136786,3297,1.763872,0,0,0,0
4,credit_card,0,0.149701,735.661402,11.074371,12.638378,677,5837.476737,23000,67.256737,4,1,0,1


In [ ]:
synthetic.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   purpose            10000 non-null  object 
 1   credit.policy      10000 non-null  int64  
 2   int.rate           10000 non-null  float64
 3   installment        10000 non-null  float64
 4   log.annual.inc     10000 non-null  float64
 5   dti                10000 non-null  float64
 6   fico               10000 non-null  int64  
 7   days.with.cr.line  10000 non-null  float64
 8   revol.bal          10000 non-null  int64  
 9   revol.util         10000 non-null  float64
 10  inq.last.6mths     10000 non-null  int64  
 11  delinq.2yrs        10000 non-null  int64  
 12  pub.rec            10000 non-null  int64  
 13  not.fully.paid     10000 non-null  int64  
dtypes: float64(6), int64(7), object(1)
memory usage: 1.1+ MB


In [ ]:
print("Real outcome distribution:")
print(df[target_col].value_counts(normalize=True))

print("\nSynthetic outcome distribution:")
print(synthetic[target_col].value_counts(normalize=True))


Real outcome distribution:
not.fully.paid
0    0.839946
1    0.160054
Name: proportion, dtype: float64

Synthetic outcome distribution:
not.fully.paid
0    0.8062
1    0.1938
Name: proportion, dtype: float64


### 5.1 Optional: condition on defaults (`not.fully.paid = 1`)

We can ask CTGAN to specifically generate records where the loan is **not fully paid**, which is useful for oversampling the rare default class.


In [ ]:
# Generate a larger number of synthetic samples with a strong bias towards the condition

large_synthetic_sample = ctgan.sample(5000)

# Filter to keep only the records where 'not.fully.paid' is 1
# Then sample 2000 from this filtered set. Using replace=True to handle cases where fewer than 2000 are initially generated.
synthetic_defaults = large_synthetic_sample[large_synthetic_sample[target_col] == 1].sample(n=2000, random_state=42, replace=True)

synthetic_defaults[target_col].value_counts(normalize=True)

,proportion
not.fully.paid,
1,1.0


## 6. Is the Synthetic Data Trustworthy?

We will evaluate the generated synthetic data set along the three pillars of quality.

1. **Fidelity**: Does the synthetic data statistically resemble the real data?
2. **Utility**: Can a machine learning algorithm trained on synthetic data perform well on real data?
3. **Privacy**: Can we guarantee that the synthetic data does not expose sentitive information from the real data?

### 6.1 Fidelity: do synthetic and real loans look statistically similar?

We evaluate fidelity for numeric columns based on individual columns and pairwise correlations

#### 6.1.1 Univariate Fidelity:

- For each numeric column, run a **Kolmogorov–Smirnov (KS) test** comparing real vs synthetic samples.
- KS statistic near 0 ⇒ distributions are similar.
- Higher KS ⇒ synthetic deviates from real.

We do this feature by feature.


In [ ]:
real = ctgan_data
syn = synthetic

ks_results = {}

for col in num_cols:
    r = real[col].sample(min(5000, len(real)), random_state=42)
    s = syn[col].sample(min(5000, len(syn)), random_state=42)
    stat, pval = ks_2samp(r, s)
    ks_results[col] = {"ks_stat": stat, "p_value": pval}

ks_df = pd.DataFrame(ks_results).T.sort_values("ks_stat")
ks_df


,ks_stat,p_value
dti,0.0522,2.410003e-06
revol.util,0.0608,1.860928e-08
revol.bal,0.0652,1.159812e-09
log.annual.inc,0.0912,1.646764e-18
installment,0.0970,6.910950e-21
pub.rec,0.0976,3.847845e-21
days.with.cr.line,0.1024,3.117596e-23
delinq.2yrs,0.1080,8.434405e-26
int.rate,0.1596,5.718661e-56
inq.last.6mths,0.1778,1.973595e-69


Interpretation:

- Which features have the **lowest** KS (best‑matched distributions)?
- Which features are hardest for CTGAN to mimic (highest KS)?



###' 6.1.2 Correlation Structure

We now compare **pairwise correlations** between numerical features in real vs synthetic data.

- Compute correlation matrices for real and synthetic numeric features.
- Look at absolute differences between them.


In [ ]:
real_corr = real[num_cols].corr()
syn_corr = syn[num_cols].corr()

corr_diff = (real_corr - syn_corr).abs()
corr_diff


,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec
int.rate,0.000000,0.139846,0.003032,0.098695,0.270106,0.142657,0.042211,0.079790,0.015148,0.054097,0.011158
installment,0.139846,0.000000,0.151998,0.164770,0.112529,0.072054,0.050546,0.216191,0.072870,0.038697,0.025286
log.annual.inc,0.003032,0.151998,0.000000,0.173093,0.064460,0.107245,0.250977,0.130884,0.074862,0.065047,0.016279
dti,0.098695,0.164770,0.173093,0.000000,0.058542,0.006216,0.004544,0.100760,0.095936,0.004769,0.025036
fico,0.270106,0.112529,0.064460,0.058542,0.000000,0.219150,0.070084,0.182638,0.108740,0.065577,0.008849
days.with.cr.line,0.142657,0.072054,0.107245,0.006216,0.219150,0.000000,0.103885,0.181830,0.088377,0.072176,0.057913
revol.bal,0.042211,0.050546,0.250977,0.004544,0.070084,0.103885,0.000000,0.034716,0.008526,0.042033,0.034132
revol.util,0.079790,0.216191,0.130884,0.100760,0.182638,0.181830,0.034716,0.000000,0.111229,0.038281,0.035033
inq.last.6mths,0.015148,0.072870,0.074862,0.095936,0.108740,0.088377,0.008526,0.111229,0.000000,0.055058,0.029967
delinq.2yrs,0.054097,0.038697,0.065047,0.004769,0.065577,0.072176,0.042033,0.038281,0.055058,0.000000,0.030230


In [ ]:
# Average absolute difference in correlation per feature
corr_diff.mean().sort_values(ascending=False)


,0
fico,0.105516
revol.util,0.101032
days.with.cr.line,0.095591
installment,0.094981
log.annual.inc,0.094352
int.rate,0.077885
dti,0.066578
inq.last.6mths,0.060065
revol.bal,0.058332
delinq.2yrs,0.042360


### 6.2 Utility: can synthetic data train a useful default model?

We evaluate **utility** using the TSTR protocol:

1. Fit a classifier on **synthetic** data.
2. Test on **held‑out real** data.
3. Compare performance with a classifier trained on **real** data (upper bound).

Metrics:

- ROC AUC
- F1‑score (for imbalanced classification)


#### 6.2.1 Create encoders/scaler on REAL training data

We will:

- Fit **OneHotEncoder** and **StandardScaler** only on the **real training** subset.
- Apply exactly the same transformations to synthetic data and real test data.


In [ ]:
# Fit on REAL TRAINING DATA ONLY
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()

ohe.fit(real_train[cat_cols])
scaler.fit(real_train[num_cols])

def preprocess_for_model(df_subset):
    X_cat = df_subset[cat_cols]
    X_num = df_subset[num_cols]
    y_out = df_subset[target_col].astype(int)

    X_cat_enc = ohe.transform(X_cat)
    X_num_scaled = scaler.transform(X_num)

    X_all = np.hstack([X_cat_enc, X_num_scaled])
    return X_all, y_out


#### 6.2.2 Baseline: Train on REAL, Test on REAL

This is our **upper bound** for performance.


In [ ]:
X_real_train, y_real_train = preprocess_for_model(real_train)
X_real_test, y_real_test = preprocess_for_model(real_test)

rf_real = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_real.fit(X_real_train, y_real_train)

y_proba_real = rf_real.predict_proba(X_real_test)[:, 1]
y_pred_real = (y_proba_real >= 0.5).astype(int)

auc_real = roc_auc_score(y_real_test, y_proba_real)
f1_real = f1_score(y_real_test, y_pred_real)

auc_real, f1_real


(np.float64(0.6691897976164206), 0.0375)

#### 6.2.3 TSTR: Train on SYNTHETIC, Test on REAL

We now:

- Use our `synthetic` dataframe as training data.
- Evaluate on the **same real test set** as above.

This tells us how good models trained purely on synthetic data are for predicting `not.fully.paid` on real loans.


In [ ]:
X_syn_train, y_syn_train = preprocess_for_model(synthetic)

rf_syn = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_syn.fit(X_syn_train, y_syn_train)

y_proba_syn = rf_syn.predict_proba(X_real_test)[:, 1]
y_pred_syn = (y_proba_syn >= 0.5).astype(int)

auc_syn = roc_auc_score(y_real_test, y_proba_syn)
f1_syn = f1_score(y_real_test, y_pred_syn)

auc_syn, f1_syn


(np.float64(0.6369879930278177), 0.13953488372093023)

#### 6.2.4 Compare utility: TRTR vs TSTR


In [ ]:
pd.DataFrame(
    {
        "AUC": [auc_real, auc_syn],
        "F1": [f1_real, f1_syn]
    },
    index=["Train REAL, Test REAL", "Train SYNTHETIC, Test REAL"]
)


,AUC,F1
"Train REAL, Test REAL",0.669190,0.037500
"Train SYNTHETIC, Test REAL",0.636988,0.139535


### 6.3 Privacy: Approximate Memorization Check

CTGAN can overfit and memorize real rows, which is a privacy risk.

A simple heuristic:
- Take numeric features from synthetic data.
- For each synthetic point, compute the distance to the **nearest real point**.
- If many synthetic points are at extremely small distance, it may indicate memorization.

This is not a formal privacy guarantee, but a useful metric.


In [ ]:
# Sample down for speed
real_num = real[num_cols].sample(5000, random_state=42)
syn_num = syn[num_cols].sample(5000, random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

distances, indices = nn.kneighbors(syn_num)
distances = distances.flatten()

pd.Series(distances).describe()


,0
count,5000.000000
mean,329.133357
std,497.104434
min,22.666635
25%,132.793025
50%,197.834994
75%,331.447574
max,10359.107491


Interpretation:

- Very small distances (e.g., many < 1e-6 after scaling) might indicate memorization.
- Larger distances suggest synthetic records are not exact copies.

For a production‑grade system, you would combine such checks with more formal privacy metrics (e.g., membership inference tests, differential privacy variants of CTGAN).


## 7. What did we learn?

In this notebook we:

1. Treated **`not.fully.paid` as the key outcome** for loan default risk.
2. Trained **CTGAN** on mixed‑type loan data to generate synthetic loans.
3. Evaluated **fidelity**:
   - KS test per numeric feature.
   - Correlation structure differences.
4. Evaluated **utility** via **Train‑Synthetic‑Test‑Real (TSTR)** against a real‑trained baseline.
5. Ran a simple **privacy heuristic** using nearest‑neighbor distances.



#Part 2 -  TVAE: Tabular Variational Autoencoder

We now train **TVAE**, another SDV model for tabular data.  
TVAE models the joint distribution using a variational autoencoder instead of an adversarial game.

We will:

1. Train TVAE on the same columns as CTGAN.
2. Generate synthetic loans.
3. Reuse the same evaluation pipeline (fidelity + TSTR utility).


In [ ]:
from sdv.single_table import TVAESynthesizer

In [ ]:
from sdv.metadata import SingleTableMetadata

tvae_data = df[cat_cols + num_cols + [target_col]].copy()

discrete_cols = cat_cols + ["inq.last.6mths", "delinq.2yrs", "pub.rec", target_col]

# Create metadata from the training data
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(tvae_data)

# Explicitly set discrete columns in the metadata
for col in discrete_cols:
    metadata.update_column(column_name=col, sdtype='categorical')

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=300,         # similar budget to CTGAN for fairness
    batch_size=500
)

tvae.fit(tvae_data)

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


### Generate synthetic loans with TVAE

We generate the same number of records (10,000) to make comparisons fair.


In [ ]:
n_synth = 10_000
synthetic_tvae = tvae.sample(n_synth)

synthetic_tvae.head()


,purpose,credit.policy,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,debt_consolidation,1,0.1121,202.43,10.948548,19.63,725,3798.895701,24623,91.91,0,0,0,0
1,debt_consolidation,1,0.1255,234.45,10.873917,18.74,691,3717.693349,14485,58.66,0,0,0,0
2,debt_consolidation,1,0.1346,591.81,11.219770,22.15,688,5188.577837,15606,74.81,0,0,0,0
3,debt_consolidation,1,0.1093,188.26,10.731861,21.35,734,3516.777846,11901,54.33,0,0,0,0
4,debt_consolidation,1,0.1343,504.73,11.307330,22.58,703,3718.770627,20111,71.11,0,0,0,0


In [ ]:
print("TVAE synthetic outcome distribution:")
print(synthetic_tvae[target_col].value_counts(normalize=True))


TVAE synthetic outcome distribution:
not.fully.paid
0    0.9997
1    0.0003
Name: proportion, dtype: float64


## Fidelity: CTGAN vs TVAE (KS test)

We compute the KS statistic per numeric feature for:

- Real vs **CTGAN** synthetic
- Real vs **TVAE** synthetic

Lower KS ⇒ closer match to real marginal distribution. [web:42][web:140]


In [ ]:
real = df[cat_cols + num_cols + [target_col]].copy()

def ks_per_feature(real_df, syn_df, num_cols):
    results = {}
    for col in num_cols:
        r = real_df[col].sample(min(5000, len(real_df)), random_state=42)
        s = syn_df[col].sample(min(5000, len(syn_df)), random_state=42)
        stat, pval = ks_2samp(r, s)
        results[col] = {"ks_stat": stat, "p_value": pval}
    return pd.DataFrame(results).T

ks_ctgan = ks_per_feature(real, synthetic, num_cols)
ks_tvae  = ks_per_feature(real, synthetic_tvae, num_cols)

ks_compare = pd.DataFrame({
    "KS_CTGAN": ks_ctgan["ks_stat"],
    "KS_TVAE": ks_tvae["ks_stat"]
}).sort_values("KS_CTGAN")

ks_compare


,KS_CTGAN,KS_TVAE
dti,0.0522,0.1408
revol.util,0.0608,0.0676
revol.bal,0.0652,0.0504
log.annual.inc,0.0912,0.0676
installment,0.0970,0.0922
pub.rec,0.0976,0.0546
days.with.cr.line,0.1024,0.1000
delinq.2yrs,0.1080,0.1178
int.rate,0.1596,0.0830
inq.last.6mths,0.1778,0.4478


You can quickly see:

- Which model better fits each numeric feature.
- Whether one model tends to systematically have lower KS across features.


### Correlation structure: CTGAN vs TVAE

We compare correlation matrices as before, now for both synthesizers. [web:143]


In [ ]:
real_corr = real[num_cols].corr()
ctgan_corr = synthetic[num_cols].corr()
tvae_corr  = synthetic_tvae[num_cols].corr()

ctgan_corr_diff = (real_corr - ctgan_corr).abs()
tvae_corr_diff  = (real_corr - tvae_corr).abs()

corr_compare = pd.DataFrame({
    "mean_abs_diff_CTGAN": ctgan_corr_diff.mean(),
    "mean_abs_diff_TVAE": tvae_corr_diff.mean()
}).sort_values("mean_abs_diff_CTGAN")

corr_compare


,mean_abs_diff_CTGAN,mean_abs_diff_TVAE
pub.rec,0.024898,NaN
delinq.2yrs,0.042360,0.054722
revol.bal,0.058332,0.071719
inq.last.6mths,0.060065,0.020179
dti,0.066578,0.111650
int.rate,0.077885,0.093549
log.annual.inc,0.094352,0.112453
installment,0.094981,0.125893
days.with.cr.line,0.095591,0.153001
revol.util,0.101032,0.099317


## Utility: TSTR for CTGAN vs TVAE

We reuse the same **Train‑Synthetic‑Test‑Real** pipeline: [web:39][web:148]

- Encoders (`ohe`) and `scaler` were fit on **real_train**.
- `preprocess_for_model` converts any dataframe to model‑ready `X`, `y`.


In [ ]:
X_tvae_train, y_tvae_train = preprocess_for_model(synthetic_tvae)

rf_tvae = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_tvae.fit(X_tvae_train, y_tvae_train)

y_proba_tvae = rf_tvae.predict_proba(X_real_test)[:, 1]
y_pred_tvae = (y_proba_tvae >= 0.5).astype(int)

auc_tvae = roc_auc_score(y_real_test, y_proba_tvae)
f1_tvae = f1_score(y_real_test, y_pred_tvae)

auc_tvae, f1_tvae


(np.float64(0.548885240392499), 0.0)

### Compare TRTR, CTGAN‑TSTR, TVAE‑TSTR


In [ ]:
utility_df = pd.DataFrame(
    {
        "AUC": [auc_real, auc_syn, auc_tvae],
        "F1":  [f1_real,  f1_syn,  f1_tvae]
    },
    index=[
        "Train REAL, Test REAL",
        "Train CTGAN, Test REAL",
        "Train TVAE, Test REAL"
    ]
)

utility_df


,AUC,F1
"Train REAL, Test REAL",0.669190,0.037500
"Train CTGAN, Test REAL",0.636988,0.139535
"Train TVAE, Test REAL",0.548885,0.000000


Discussion prompts:

- Which synthesizer gives a classifier whose AUC/F1 is closer to the **real‑trained** baseline?
- Are there differences in calibration or class balance that might explain performance? [web:148][web:143]


## Privacy heuristic: nearest‑neighbor distance (CTGAN vs TVAE)

We reuse the **nearest‑neighbor distance** approach to compare memorization risk for the two models. [web:140][web:146]


In [ ]:
# Sample real numeric subset for reference
real_num = real[num_cols].sample(5000, random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

# CTGAN
syn_ctgan_num = synthetic[num_cols].sample(5000, random_state=42)
dist_ctgan, _ = nn.kneighbors(syn_ctgan_num)
dist_ctgan = dist_ctgan.flatten()

# TVAE
syn_tvae_num = synthetic_tvae[num_cols].sample(5000, random_state=42)
dist_tvae, _ = nn.kneighbors(syn_tvae_num)
dist_tvae = dist_tvae.flatten()

privacy_df = pd.DataFrame(
    {
        "CTGAN_dist": dist_ctgan,
        "TVAE_dist": dist_tvae
    }
)

privacy_df.describe()


,CTGAN_dist,TVAE_dist
count,5000.000000,5000.000000
mean,329.133357,236.936906
std,497.104434,411.501663
min,22.666635,10.332692
25%,132.793025,113.971904
50%,197.834994,160.919179
75%,331.447574,242.321412
max,10359.107491,14301.401818


Interpretation:

- Higher typical nearest‑neighbor distances suggest less memorization.
- If one model has consistently much smaller distances, it may be overfitting more to specific real points. [web:140][web:146]


## CTGAN vs TVAE: what to observe


1. **Fidelity (KS & correlation):**
   - For which features does CTGAN best mimic the real distribution?
   - Where does TVAE do better?
   - Is one model more consistent across features?

2. **Utility (TSTR AUC & F1):**
   - Which synthetic dataset produces more useful classifiers for predicting `not.fully.paid`?
   - How close are they to the real‑trained baseline?

3. **Privacy heuristic (NN distances):**
   - Does either model appear to memorize real records more?

This gives a concrete, model‑agnostic way to discuss **fidelity–utility–privacy trade‑offs** across two popular tabular synthesizers on a realistic loan default problem.


#Required Task 18
Load the file `fraud_transactions.csv` and create a synthetic data set of 5000 records. Evaluate the quality of the synthetic data created.

## Task 18: Generate and Evaluate Synthetic Data for `fraud_transactions.csv`

### 1. Load and Inspect `fraud_transactions.csv`

In [ ]:
fraud_df = pd.read_csv('/content/fraud_transactions.csv')

print("First 5 rows of fraud_df:")
display(fraud_df.head())

print("\nInfo about fraud_df:")
fraud_df.info()

print("\nDescription of fraud_df numerical columns:")
display(fraud_df.describe())

First 5 rows of fraud_df:


,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,2/27/19 21:32,"fraud_Langosh, Wintheiser and Hyatt",food_dining,83.64,F,TX,"Physicist, medical",0
1,2/13/19 19:41,fraud_Dibbert and Sons,entertainment,79.13,M,PA,Secretary/administrator,0
2,1/11/19 20:03,"fraud_McDermott, Osinski and Morar",home,12.02,F,CA,"Buyer, industrial",0
3,1/20/19 9:08,fraud_Bauch-Raynor,grocery_pos,84.41,M,TN,Clothing/textile technologist,0
4,1/4/19 17:04,"fraud_Reichert, Huels and Hoppe",shopping_net,2.81,F,ME,Financial trader,0



Info about fraud_df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trans_date_trans_time  6486 non-null   object 
 1   merchant               6486 non-null   object 
 2   category               6486 non-null   object 
 3   amt                    6486 non-null   float64
 4   gender                 6486 non-null   object 
 5   state                  6486 non-null   object 
 6   job                    6486 non-null   object 
 7   is_fraud               6486 non-null   int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 405.5+ KB

Description of fraud_df numerical columns:


,amt,is_fraud
count,6486.000000,6486.000000
mean,101.700956,0.074931
std,189.292632,0.263300
min,1.000000,0.000000
25%,12.572500,0.000000
50%,52.080000,0.000000
75%,91.945000,0.000000
max,2312.210000,1.000000


### 2. Preprocessing `fraud_df`

In [ ]:
# Drop 'trans_date_trans_time' as it requires specialized handling for synthesis that is outside the scope of this example
fraud_df = fraud_df.drop(columns=['trans_date_trans_time'])

# Define target, categorical, and numerical columns
fraud_target_col = "is_fraud"

fraud_cat_cols = [
    "merchant",
    "category",
    "gender",
    "state",
    "job"
]

fraud_num_cols = [
    "amt"
]

# Ensure target column is treated as categorical for SDV models
fraud_discrete_cols = fraud_cat_cols + [fraud_target_col]

# Display target distribution
print("\nDistribution of fraud_target_col:")
print(fraud_df[fraud_target_col].value_counts(normalize=True))


Distribution of fraud_target_col:
is_fraud
0    0.925069
1    0.074931
Name: proportion, dtype: float64


### 3. Train TVAE to generate synthetic fraud data

In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

tvae_fraud_data = fraud_df.copy()

# Create metadata from the training data
metadata_fraud = SingleTableMetadata()
metadata_fraud.detect_from_dataframe(tvae_fraud_data)

# Explicitly set discrete columns in the metadata
for col in fraud_discrete_cols:
    metadata_fraud.update_column(column_name=col, sdtype='categorical')

# Initialize and train TVAE
tvae_fraud = TVAESynthesizer(
    metadata=metadata_fraud,
    epochs=300,
    batch_size=500
)

tvae_fraud.fit(tvae_fraud_data)

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


### 4. Generate synthetic fraud data

In [ ]:
n_synth_fraud = 5000
synthetic_fraud_tvae = tvae_fraud.sample(n_synth_fraud)

print("First 5 rows of synthetic fraud data:")
display(synthetic_fraud_tvae.head())

print("\nSynthetic fraud outcome distribution:")
print(synthetic_fraud_tvae[fraud_target_col].value_counts(normalize=True))

First 5 rows of synthetic fraud data:


,merchant,category,amt,gender,state,job,is_fraud
0,fraud_Schuppe LLC,entertainment,12.05,F,IN,Senior tax professional/tax inspector,0
1,fraud_Waelchi Inc,food_dining,82.71,F,FL,Podiatrist,0
2,"fraud_McDermott, Osinski and Morar",home,20.32,F,OH,Magazine features editor,0
3,fraud_Waelchi Inc,food_dining,11.05,M,CA,Advertising account planner,0
4,fraud_Kuhn LLC,shopping_net,1.00,F,ME,Film/video editor,0



Synthetic fraud outcome distribution:
is_fraud
0    0.947
1    0.053
Name: proportion, dtype: float64


### 5. Evaluate Fidelity of Synthetic Fraud Data (TVAE)

#### 5.1 Univariate Fidelity (KS Test)

In [ ]:
def ks_per_feature_single(real_df, syn_df, num_cols):
    results = {}
    for col in num_cols:
        # Sample down for speed if datasets are large
        r = real_df[col].sample(min(5000, len(real_df)), random_state=42)
        s = syn_df[col].sample(min(5000, len(syn_df)), random_state=42)
        stat, pval = ks_2samp(r, s)
        results[col] = {"ks_stat": stat, "p_value": pval}
    return pd.DataFrame(results).T.sort_values("ks_stat")

real_fraud_data = fraud_df.copy()
ks_fraud_tvae = ks_per_feature_single(real_fraud_data, synthetic_fraud_tvae, fraud_num_cols)

print("KS Test Results for TVAE Synthetic Fraud Data:")
print(ks_fraud_tvae)

KS Test Results for TVAE Synthetic Fraud Data:
     ks_stat       p_value
amt    0.165  8.269381e-60


#### 5.2 Correlation Structure

In [ ]:
real_fraud_corr = real_fraud_data[fraud_num_cols].corr()
syn_fraud_tvae_corr = synthetic_fraud_tvae[fraud_num_cols].corr()

corr_fraud_diff_tvae = (real_fraud_corr - syn_fraud_tvae_corr).abs()

print("Absolute Difference in Correlation Matrix (Real vs TVAE Synthetic Fraud Data):")
display(corr_fraud_diff_tvae)

print("\nAverage absolute difference in correlation per feature (TVAE Synthetic Fraud Data):")
print(corr_fraud_diff_tvae.mean().sort_values(ascending=False))

Absolute Difference in Correlation Matrix (Real vs TVAE Synthetic Fraud Data):


,amt
amt,0.0



Average absolute difference in correlation per feature (TVAE Synthetic Fraud Data):
amt    0.0
dtype: float64


### 6. Evaluate Utility of Synthetic Fraud Data (TVAE)

#### 6.1 Create encoders/scaler on REAL training data

In [ ]:
# First, split the real data into train and test for utility evaluation
real_fraud_train, real_fraud_test = train_test_split(
    fraud_df,
    test_size=0.2,
    random_state=42,
    stratify=fraud_df[fraud_target_col]
)

# Fit OneHotEncoder and StandardScaler only on the REAL TRAINING data
ohe_fraud = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler_fraud = StandardScaler()

ohe_fraud.fit(real_fraud_train[fraud_cat_cols])
scaler_fraud.fit(real_fraud_train[fraud_num_cols])

def preprocess_for_fraud_model(df_subset):
    X_cat = df_subset[fraud_cat_cols]
    X_num = df_subset[fraud_num_cols]
    y_out = df_subset[fraud_target_col].astype(int)

    X_cat_enc = ohe_fraud.transform(X_cat)
    X_num_scaled = scaler_fraud.transform(X_num)

    X_all = np.hstack([X_cat_enc, X_num_scaled])
    return X_all, y_out

#### 6.2 Baseline: Train on REAL, Test on REAL

In [ ]:
X_real_fraud_train, y_real_fraud_train = preprocess_for_fraud_model(real_fraud_train)
X_real_fraud_test, y_real_fraud_test = preprocess_for_fraud_model(real_fraud_test)

rf_real_fraud = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_real_fraud.fit(X_real_fraud_train, y_real_fraud_train)

y_proba_real_fraud = rf_real_fraud.predict_proba(X_real_fraud_test)[:, 1]
y_pred_real_fraud = (y_proba_real_fraud >= 0.5).astype(int)

auc_real_fraud = roc_auc_score(y_real_fraud_test, y_proba_real_fraud)
f1_real_fraud = f1_score(y_real_fraud_test, y_pred_real_fraud)

print(f"Real-trained model AUC: {auc_real_fraud:.4f}, F1: {f1_real_fraud:.4f}")

Real-trained model AUC: 0.9831, F1: 0.8343


#### 6.3 TSTR: Train on SYNTHETIC, Test on REAL

In [ ]:
X_syn_fraud_train, y_syn_fraud_train = preprocess_for_fraud_model(synthetic_fraud_tvae)

rf_syn_fraud = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_syn_fraud.fit(X_syn_fraud_train, y_syn_fraud_train)

y_proba_syn_fraud = rf_syn_fraud.predict_proba(X_real_fraud_test)[:, 1]
y_pred_syn_fraud = (y_proba_syn_fraud >= 0.5).astype(int)

auc_syn_fraud = roc_auc_score(y_real_fraud_test, y_proba_syn_fraud)
f1_syn_fraud = f1_score(y_real_fraud_test, y_pred_syn_fraud)

print(f"Synthetic-trained model AUC: {auc_syn_fraud:.4f}, F1: {f1_syn_fraud:.4f}")

Synthetic-trained model AUC: 0.9347, F1: 0.6081


#### 6.4 Compare Utility

In [ ]:
pd.DataFrame(
    {
        "AUC": [auc_real_fraud, auc_syn_fraud],
        "F1": [f1_real_fraud, f1_syn_fraud]
    },
    index=["Train REAL, Test REAL", "Train SYNTHETIC, Test REAL"]
)

,AUC,F1
"Train REAL, Test REAL",0.983064,0.834286
"Train SYNTHETIC, Test REAL",0.934706,0.608108


### 7. Evaluate Privacy of Synthetic Fraud Data (TVAE)

In [ ]:
# Sample down for speed
real_fraud_num = real_fraud_data[fraud_num_cols].sample(min(5000, len(real_fraud_data)), random_state=42)
syn_fraud_num = synthetic_fraud_tvae[fraud_num_cols].sample(min(5000, len(synthetic_fraud_tvae)), random_state=42)

nn_fraud = NearestNeighbors(n_neighbors=1)
nn_fraud.fit(real_fraud_num)

distances_fraud, _ = nn_fraud.kneighbors(syn_fraud_num)
distances_fraud = distances_fraud.flatten()

pd.Series(distances_fraud).describe()

,0
count,5000.000000
mean,0.136924
std,1.262439
min,0.000000
25%,0.000000
50%,0.010000
75%,0.020000
max,64.200000


In [ ]:
# First, split the real data into train and test for utility evaluation
real_fraud_train, real_fraud_test = train_test_split(
    fraud_df,
    test_size=0.2,
    random_state=42,
    stratify=fraud_df[fraud_target_col]
)

# Fit OneHotEncoder and StandardScaler only on the REAL TRAINING data
ohe_fraud = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler_fraud = StandardScaler()

ohe_fraud.fit(real_fraud_train[fraud_cat_cols])
scaler_fraud.fit(real_fraud_train[fraud_num_cols])

def preprocess_for_fraud_model(df_subset):
    X_cat = df_subset[fraud_cat_cols]
    X_num = df_subset[fraud_num_cols]
    y_out = df_subset[fraud_target_col].astype(int)

    X_cat_enc = ohe_fraud.transform(X_cat)
    X_num_scaled = scaler_fraud.transform(X_num)

    X_all = np.hstack([X_cat_enc, X_num_scaled])
    return X_all, y_out